# Frozen InternVideo2 visual features added to the winning graph + TCN

Use existing labels, audio features, graph preprocessing and saved controls.
Five seed-0 whole-window folds; final three seeds; 60 fixed epochs. Train only
the small possession/fusion network. Video encoder weights remain frozen.
Initial diagnostic features cover every existing labelled window plus context;
full-match expansion is conditional on a demonstrated gain and separately scored.
No new annotations or pretrained weights in output artifacts.


In [ ]:
from pathlib import Path
import hashlib,json,os,subprocess,sys,time
import torch
assert torch.cuda.is_available()
print('GPU:',torch.cuda.get_device_name(0),'count',torch.cuda.device_count(),flush=True)
INPUT=Path('/kaggle/input');WORK=Path('/kaggle/working/possession_vision')
WORK.mkdir(parents=True,exist_ok=True)
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
roots=[p.parent for p in INPUT.rglob('VISION_EXPERIMENT.md') if (p.parent/'visual_possession/experiment.py').is_file()]
assert len(roots)==1
CODE=roots[0]
configs=list(INPUT.rglob('vision_study_input.json'));assert len(configs)==1
FILES=configs[0].parent;plan=json.loads(configs[0].read_text())
labels=list(INPUT.rglob('relational_input.json'));assert len(labels)==1
LABELS=labels[0].parent
base=[p for p in INPUT.rglob('features.npz') if sha(p)==plan['feature_sha256']]
audio=[p for p in INPUT.rglob('audio_cache.npz') if sha(p)==plan['audio_sha256']]
assert len(base)==len(audio)==1
FEATURES,AUDIO,GRAPH=base[0],audio[0],LABELS/'graph.npz'
assert sha(GRAPH)==plan['graph_sha256']
GEOMETRY=FILES/'geometry.npz';assert sha(GEOMETRY)==plan['geometry_sha256']
for split in ('train','validation'):assert sha(LABELS/f'{split}.csv')==plan[f'{split}_sha256']
sources=[p.parent for p in INPUT.rglob('experiment.json') if json.loads(p.read_text()).get('kind')=='team_symmetric_graph_memory_span_ablation']
assert len(sources)==1
SOURCE=sources[0]
videos=list(INPUT.rglob('BAR-ATH_tactical.mp4'));assert len(videos)==1
VIDEO=videos[0]
DEPS=Path('/tmp/barca_vision_python')
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check',
               '--target',str(DEPS),'-r',str(CODE/'requirements-temporal.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check',
               '--target',str(DEPS),'--no-deps','-r',str(CODE/'requirements-foundation.txt')],check=True)
env=dict(os.environ,PYTHONPATH=str(DEPS)+os.pathsep+str(CODE),PYTHONDONTWRITEBYTECODE='1',
         HF_HOME='/tmp/barca_vision_hf',HF_HUB_DISABLE_XET='1',HF_HUB_DISABLE_TELEMETRY='1',
         HF_HUB_DOWNLOAD_TIMEOUT='90',HF_HUB_ETAG_TIMEOUT='30',HF_HUB_DISABLE_PROGRESS_BARS='1',
         CUBLAS_WORKSPACE_CONFIG=':4096:8')
env.pop('HF_HUB_OFFLINE',None)
def run(*args):subprocess.run([sys.executable,*map(str,args)],cwd=CODE,env=env,check=True)
run('-m','unittest','discover','-s','tests','-v')
print('Full suite passed; beginning frozen feature extraction.',flush=True)


In [ ]:
EXTRACT=WORK/'diagnostic_features'
run('-m','visual_possession','extract','--video',VIDEO,'--geometry',GEOMETRY,
    '--model-cache','/tmp/barca_vision_models','--out',EXTRACT,'--intervals',FILES/'extraction_intervals.json')
VISION=EXTRACT/'visual_cache.npz'


In [ ]:
RUN=WORK/'study'
started=time.monotonic()
run('-m','visual_possession','compare','--features',FEATURES,'--audio',AUDIO,'--graph',GRAPH,
    '--vision',VISION,'--train-labels',LABELS/'train.csv','--validation-labels',LABELS/'validation.csv',
    '--source-study',SOURCE,'--out',RUN,'--device','cuda','--epochs','60','--seeds','0','1','2','--bootstrap','1000')


In [ ]:
checks={}
for calibrated in (False,True):
    filename='calibrated.csv' if calibrated else 'predictions.csv'
    output=RUN/'global/graph_vision'/('regenerated_'+filename)
    args=['-m','visual_possession','predict','--features',FEATURES,'--audio',AUDIO,'--graph',GRAPH,
          '--vision',VISION,'--model',RUN/'global/graph_vision','--out',output,'--device','cuda']
    if calibrated:args.append('--calibrated')
    run(*args)
    assert output.read_bytes()==(RUN/'global/graph_vision'/filename).read_bytes()
    checks[filename]=sha(output)
    print('Byte-identical vision regeneration:',filename,flush=True)
source={str(p.relative_to(CODE)):sha(p) for root in ('visual_possession','relational_possession','temporal_possession','learned_possession','structured_possession') for p in (CODE/root).glob('*.py')}
protocol=dict(torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),
              source_sha256=source,regeneration=checks,all_outputs_regenerated=True,
              private_inputs=plan,vision_cache_sha256=sha(VISION),audio_encoder_unchanged=True,
              references_unchanged=all(sha(LABELS/f'{split}.csv')==plan[f'{split}_sha256'] for split in ('train','validation')))
assert protocol['references_unchanged']
(WORK/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')
report=json.loads((RUN/'report.json').read_text())
baseline=report['pooled_oof']['graph_control']['spell_f1_mean']
new=report['pooled_oof']['graph_vision']['spell_f1_mean']
control_val=report['scores']['graph_control']['spell_f1_mean']
vision_val=report['scores']['graph_vision_calibrated']['spell_f1_mean']
expand=bool(new>baseline and vision_val>=control_val)
(WORK/'expansion_decision.json').write_text(json.dumps(dict(expand=expand,training_oof_control=baseline,
    training_oof_vision=new,validation_control=control_val,validation_vision=vision_val,
    criterion='Expand only if diagnostic training OOF spellF1 improves and reused validation does not regress'),indent=2)+'\n')
print('FULL-MATCH EXPANSION RECOMMENDED:',expand,flush=True)
print('FROZEN VISION GPU STUDY COMPLETE',flush=True)
